In [1]:


import json
from datetime import date
from collections import defaultdict

# 1. INPUT DATA (as provided)
input_data = [
    {
        "accStatus": "N",
        "amount": "139",
        "balance": "500000",
        "debtGroupAcc": "1",
        "interestType": "1",
        "rate": "12",
        "tranDate": "2025-03-02"
    },
    {
        "accStatus": "N",
        "amount": "139",
        "balance": "500000",
        "debtGroupAcc": "1",
        "interestType": "1",
        "rate": "12",
        "tranDate": "2025-03-03"
    },
    {
        "accStatus": "N",
        "amount": "139",
        "balance": "500000",
        "debtGroupAcc": "1",
        "interestType": "2",
        "rate": "6",
        "tranDate": "2025-03-02"
    },
    {
        "accStatus": "N",
        "amount": "139",
        "balance": "500000",
        "debtGroupAcc": "1",
        "interestType": "2",
        "rate": "6",
        "tranDate": "2025-03-03"
    },
    {
        "accStatus": "N",
        "amount": "139",
        "balance": "500000",
        "debtGroupAcc": "1",
        "interestType": "3",
        "rate": "10",
        "tranDate": "2025-03-02"
    },
    {
        "accStatus": "N",
        "amount": "139",
        "balance": "500000",
        "debtGroupAcc": "1",
        "interestType": "3",
        "rate": "10",
        "tranDate": "2025-03-03"
    }
    # Add more realistic data here if available,
    # especially with varying amounts and dates spanning the periods
]

# 2. DEFINE OUTPUT PERIODS (Based on your desired output structure)
# Note: Corrected the likely typo in the first FromDate ("2025-03-020" -> "2025-03-21")
#       Assuming periods are consecutive and non-overlapping. Adjust if needed.
periods = [
    {
        # Period 1: Corresponds to the second object in your example output
        "FromDate": "2025-03-02",
        "ToDate": "2025-03-20",
        "balance": "500000",
        "rate": "12"
    },
    {
        # Period 2: Corresponds to the first object in your example output
        # Adjusted FromDate assuming non-overlapping ranges
        "FromDate": "2025-03-21",
        "ToDate": "2025-04-02",
        "balance": "500000",
        "rate": "10"
    }
]

# 3. INITIALIZE RESULT LIST
final_output = []

# 4. PROCESS EACH PERIOD
for period in periods:
    print(f"--- Processing Period: {period['FromDate']} to {period['ToDate']} ---")

    # Use datetime objects for reliable comparison
    try:
        period_from_date = date.fromisoformat(period['FromDate'])
        period_to_date = date.fromisoformat(period['ToDate'])
    except ValueError as e:
        print(f"Error parsing period dates: {e}. Skipping period.")
        continue

    # Use defaultdict to easily sum amounts by interestType
    amounts_by_type = defaultdict(float)

    # 5. FILTER input data for the current period and AGGREGATE
    records_in_this_period = 0
    for record in input_data:
        try:
            tran_date = date.fromisoformat(record['tranDate'])

            # Check if the transaction date falls within the period (inclusive)
            if period_from_date <= tran_date <= period_to_date:
                records_in_this_period += 1
                interest_type = record['interestType']
                # Convert amount string to float for summation
                try:
                    amount = float(record.get('amount', 0)) # Use .get for safety
                except (ValueError, TypeError):
                    amount = 0.0 # Handle cases where amount is missing or not a number

                amounts_by_type[interest_type] += amount

        except ValueError as e:
            print(f"Warning: Skipping record due to invalid date format: {record.get('tranDate')}, Error: {e}")
        except Exception as e:
            print(f"Warning: Skipping record due to unexpected error: {e}, Record: {record}")


    print(f"Found {records_in_this_period} records in this period.")
    print(f"Aggregated amounts (before formatting): {dict(amounts_by_type)}") # Show intermediate sums

    # 6. FORMAT the 'Data' array for the output object
    data_list = []
    # Sort by interestType for consistent output order
    for interest_type in sorted(amounts_by_type.keys()):
        # Convert the summed amount back to a string for the output
        # Using int() to remove decimal if amounts are whole numbers
        total_amount_str = str(int(amounts_by_type[interest_type]))
        data_list.append({
            "interestType": interest_type,
            "totalamount": total_amount_str
        })

    # 7. CONSTRUCT the output object for this period
    output_object = {
        "FromDate": period['FromDate'], # Use original string dates for output
        "ToDate": period['ToDate'],
        "balance": period['balance'],
        "rate": period['rate'],
        "Data": data_list
    }

    # 8. ADD the result for this period to the final list
    final_output.append(output_object)
    print("-" * 40)


# 9. PRINT THE FINAL RESULT (formatted as JSON)
print("\n=== FINAL OUTPUT ===\n")
print(json.dumps(final_output, indent=4))

# --- Explanation of Discrepancy ---
print("\n=== NOTE ON NUMERIC VALUES ===")
print("The 'totalamount' values in the output above are calculated by SUMMING the 'amount' field ('139')")
print("from the input records that fall within each defined period and match the 'interestType'.")
print("These sums WILL NOT MATCH the large numbers ('100000', '139000', etc.) shown in your EXAMPLE output,")
print("because the provided input data's 'amount' field does not support those large resulting sums.")
print("The code implements the requested *logic* (filtering by date, grouping by type, summing amount),")
print("but the example input/output numeric values are inconsistent.")

--- Processing Period: 2025-03-02 to 2025-03-20 ---
Found 6 records in this period.
Aggregated amounts (before formatting): {'1': 278.0, '2': 278.0, '3': 278.0}
----------------------------------------
--- Processing Period: 2025-03-21 to 2025-04-02 ---
Found 0 records in this period.
Aggregated amounts (before formatting): {}
----------------------------------------

=== FINAL OUTPUT ===

[
    {
        "FromDate": "2025-03-02",
        "ToDate": "2025-03-20",
        "balance": "500000",
        "rate": "12",
        "Data": [
            {
                "interestType": "1",
                "totalamount": "278"
            },
            {
                "interestType": "2",
                "totalamount": "278"
            },
            {
                "interestType": "3",
                "totalamount": "278"
            }
        ]
    },
    {
        "FromDate": "2025-03-21",
        "ToDate": "2025-04-02",
        "balance": "500000",
        "rate": "10",
        "Data": 